# Paper 17 · The Integrated Character Library

Build and qualify **one character voice + one accent + one emotion** as a reusable recipe. Keep the main voice-design app unchanged until this prototype proves the complete path.

**Start here:** Run all cells to activate the configuration controls. These cells only build a plan and inspect local evidence. They do not generate audio, process recordings, approve assets, install models, or change the existing app.

The controls express what we want. A dropdown label is never proof that the audio actually delivers it.

In [1]:
from pathlib import Path
import sys, json, html
import ipywidgets as w
from IPython.display import display, HTML
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'services/speaker-studio/accent_pipeline.py').exists())
sys.path.insert(0, str(ROOT / 'knowledge/research/paper-17-study'))
from character_library import Intent, CHARACTERS, ACCENTS, EMOTIONS, plan, passport_template, location
PRESET = json.loads((ROOT / 'services/speaker-studio/speech-processing-preset.json').read_text())['settings']
print('Design prototype ready. Audio generation and processing remain disconnected.')

Design prototype ready. Audio generation and processing remain disconnected.


## Configure the character

Select the identity, accent and emotion independently. **Develop** is the checkpoint path for building a new combination. **Use** becomes available only after that exact profile passes qualification; direct single-pass reuse needs its own test.

First integrated target: **Doctor Doom + Indian English + Anger**. Indian accent transfer has prior evidence. Combining it with anger is the new experiment. Russian remains available for diagnostic work; do not generalize the recent Victor result into full qualification.

In [2]:
voice = w.Dropdown(options=[(v['name'], k) for k,v in CHARACTERS.items()], value='doom', description='Character:')
accent = w.Dropdown(options=[(v['label'], k) for k,v in ACCENTS.items()], value='indian', description='Accent:')
emotion = w.Dropdown(options=list(EMOTIONS), value='Anger', description='Emotion:')
mode = w.ToggleButtons(options=['Develop','Use'], value='Develop', description='Mode:')
text = w.Textarea(value='You mistake my patience for weakness.', description='Words:', layout=w.Layout(width='100%',height='75px'))
summary = w.HTML(); flow = w.HTML(); passport = w.HTML()
def refresh(change=None):
    try:
        intent = Intent(voice.value,accent.value,emotion.value,text.value,mode.value)
        result = plan(intent)
        esc = html.escape
        summary.value = f"<p><strong>{esc(result['status'])}</strong></p><p>{result['word_count']} / 35 words. Accent source: {esc(location(accent.value))}.</p>"
        if result['stages']:
            flow.value = ''.join(f"<section style='border-top:1px solid #89949d;padding:12px 0'><h3>{i+1}. {esc(s['name'])}</h3><p>{esc(s['instruction'])}</p><p><strong>Continue when:</strong> {esc(s['gate'])}</p>" + (f"<p><strong>Not proven yet:</strong> {esc(s['blocker'])}</p>" if s.get('blocker') else '') + '</section>' for i,s in enumerate(result['stages']))
        else:
            flow.value = '<p>No qualified combined profiles have been imported. Existing single-dimension successes are not automatically promoted.</p>'
        passport.value = '<details><summary>Profile passport structure</summary><pre style="white-space:pre-wrap">'+esc(json.dumps(passport_template(intent),indent=2))+'</pre></details>'
    except ValueError as exc:
        summary.value = '<p><strong>'+html.escape(str(exc))+'</strong></p>'; flow.value=''; passport.value=''
for control in (voice,accent,emotion,mode,text): control.observe(refresh,names='value')
workbench = w.VBox([w.HBox([voice,accent,emotion],layout=w.Layout(flex_flow='row wrap')),mode,text,summary,flow,passport])
refresh(); display(workbench)

## One vertical process, horizontal audio development

| Checkpoint | Original | Processed output | What moves downward |
|---|---|---|---|
| Real performance reference | Untouched recording, source metadata and exact transcript | Optional Resemble → Pedalboard output | The version the user selects |
| Actor saying the requested words | Raw generated actor take | One optional processed output | The exact accepted actor asset |
| Character conversion | Raw conversion into the pinned identity | One optional processed output | The exact accepted character take |

At generated checkpoints the main actions remain **Regenerate** and **Accept & continue**. Regenerate uses a new seed for that checkpoint only. Reprocessing replaces the one visible processed output, never the original. Replaced assets remain traceable if another result depends on them.

Changing an upstream selection makes downstream acceptance stale. Existing audio is preserved with its real lineage. A processed input must never be relabeled Original in a later attempt; the original source and the frozen generation input are separate fields.

## How accent and emotion meet

A performance recording carries pronunciation, emotion, timing and a speaker identity together. We cannot assume that attaching an accent file and an emotion file will combine those properties correctly.

**Preferred development route:** select a real performance that demonstrates both the chosen accent and emotion, generate the new words, approve that actor, then convert the accepted performance into the character.

**Experimental route if no matched performance exists:** test whether the actor model can preserve the real accent reference while applying an emotion direction. Review the actor before conversion. If it stays neutral or loses the accent, stop at that checkpoint; change the reference or supported generation method. Do not compensate by changing the final character identity.

Neither route is yet qualified for the default Indian-English angry Doom combination. We will compare the actor against its reference and Doom against that exact actor. A word check alone never approves emotion, accent or identity.

## Processing defaults

Restore and Polish are independently enabled. The current **Speech clarity** recipe is a starting configuration, not a universal optimum. Adjust it when listening exposes artifacts. Always process a copy of the stage's original; applying a new recipe must not silently compound the previous effects.

In [3]:
display(HTML('<table>'+''.join('<tr><th style="text-align:left;padding:5px">'+html.escape(k)+'</th><td>'+html.escape(str(PRESET[k]))+'</td></tr>' for k in ['preset','restore','polish','evaluations','solver','denoise','temperature','profile','target_lufs','true_peak_db','sample_rate'])+'</table>'))

preset,speech-clarity-v1
restore,True
polish,True
evaluations,64
solver,midpoint
denoise,0.1
temperature,0.5
profile,Gentle
target_lufs,-20.0
true_peak_db,-2.0
sample_rate,44100


## What is already supported, and what is new

| Evidence | Meaning |
|---|---|
| Santa anger: three lines passed user listening | Evidence for that saved voice and anger recipe only |
| Real British, American and Indian references transferred into Doom | Accent-only evidence for the tested recordings and lines |
| Russian preprocessing helped the Victor opening | Useful local diagnostic result; still inspect accent and final quality |
| Speech clarity processing ran and passed technical checks | Valid audio output; not proof that every voice sounds better |
| Accent + emotion together | **New work. Not qualified yet** |
| A saved combined profile generating directly in one pass | **Separate optimization test. Not qualified yet** |

Source locations remain optional. For ASI, Hindi is documented as native language; country and locality remain unknown rather than being guessed.

## The qualification gate

Use three fresh sentences, each at most 35 words, with the same pinned recipe. For **each** sentence record five listening verdicts: correct words, same character, chosen accent, chosen emotion, acceptable audio quality. All five must pass on all three lines. Automatic transcription and audio-integrity checks support this decision but do not replace listening.

Regenerating one failure does not erase it from the passport. Save a profile only when its accepted assets, model versions, per-stage seeds, processing parameters and qualification evidence are complete.

In [4]:
# Executable contract checks; no audio engines are called.
from character_library import qualify
verdict = dict(words=True, identity=True, accent=True, emotion=True, quality=True)
assert qualify([verdict, verdict, verdict])
assert not qualify([verdict, verdict, dict(verdict, accent=False)])
assert not plan(Intent(mode='Use'))['ready_for_future_generation']
assert not plan(Intent())['can_generate']
print('Qualification contracts pass. No voice profile has been approved by these checks.')

Qualification contracts pass. No voice profile has been approved by these checks.


## Prototype build order

1. **This notebook:** agree on the input contract, visible checkpoints, immutable assets and acceptance rules. Configuration controls and qualification rules are executable; generation is not connected yet.
2. **Connect one combination:** Doctor Doom + Indian English + Anger. Reuse the existing reference catalog, processing worker, actor generation and conversion adapters. Add the missing combined-performance trial and capture every intermediate recording.
3. **Qualify three new lines:** record the user's five listening verdicts per line. Keep an explicit failure path to the weak checkpoint.
4. **Save and reuse the profile:** new text runs the verified recipe without repeating setup. Keep the checkpoint view accessible. Test direct single-pass generation separately before enabling it.
5. **Upgrade the app:** reuse the same library/profile service and contracts behind the app's Character / Accent / Emotion selectors. Do not build an independent second pipeline.

**Promotion decision:** no app integration until the integrated notebook path passes the combined listening test and survives save/reopen with original, selected input, results and passport intact.

## Existing evidence and implementation anchors

The design is grounded in the current repository, not a new model recommendation:

- Paper 16 and `knowledge/research/paper-16-study/` contain accent, emotion and processing evidence.
- `services/speaker-studio/accent_pipeline.py` holds durable checkpoints, asset integrity, explicit acceptance and independent retries.
- `services/speaker-studio/accent_processing.py` connects the existing local Resemble Enhance and Spotify Pedalboard worker.
- `speech-processing-preset.json` supplies the adjustable Speech clarity default.
- `original-reference-repair-verification.json` records the correction separating canonical raw references from frozen generation inputs.
- `paper-17-study/character_library.py` is this notebook's design-only planner and passport template. It does not call audio engines.

The current app and accent listening workspace remain separate and available. No services are stopped or replaced by running this notebook.